In [28]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import pandas as pd
import numpy as np
import random
import gc
import os
import time
import dask.array as da
from joblib import Parallel, delayed
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import gamma  
from IPython.display import display
from dask import delayed

start_time = time.time()

In [30]:
# Credit Risk Parameters (See Gordy Model or section 3 of my thesis: Transition Risk Add-on: a simulation approach. pdf for details)
mean = 1
variance = 4
k = mean ** 2 / variance
theta = variance / mean
E_LGD = 0.50
sigma = 2
lambda_ = 0.5
eta = 0.25
sigma2 = sigma ** 2

#Climate Risk Parameters
model = 'WITCH' #Possible Choices: REMIND, WITCH
scenario = 'LIMITS450' #Possible Choices: LIMITSOilIndependence, LIMITS450, LIMITS500, LIMITSPledges, LIMITSEnergyIndependence
Selected_Region = 'Europe' #Possible Choices: Asia, Europe, LAM, NAM, ROW, Africa, World
num_simulations = 10000 #Number of simulations to run / Number of Monte Carlo iterations / Number of random portfolios to generate
max_subsectors_per_sector = 3 #Max number of subsectors to consider for each sector
sectors_data = {
    'primary_fossil': ['Primary Energy|Coal', 'Primary Energy|Gas', 'Primary Energy|Oil'],
    'secondary_fossil': ['Secondary Energy|Electricity|Coal', 'Secondary Energy|Electricity|Gas', 'Secondary Energy|Electricity|Oil'],
    'secondary_renewable': ['Secondary Energy|Electricity|Biomass', 'Secondary Energy|Electricity|Geothermal',
                            'Secondary Energy|Electricity|Solar|CSP', 'Secondary Energy|Electricity|Nuclear',
                            'Secondary Energy|Electricity|Solar|PV', 'Secondary Energy|Electricity|Wind', 'Secondary Energy|Electricity|Hydro']
}
max_percentages = { #Max exposure for each energy category
    'primary_fossil': 50,
    'secondary_fossil': 50,
    'secondary_renewable': 0
}
risk_free_rate = 0.03 #Risk-free rate for discounting climate losses
CONFIDENCE_LEVEL = 0.995


In [31]:
folder = 'data'
file_portfolio = f"portafoglio_{Selected_Region}.csv"
file_shocks = f"shock_{Selected_Region}.csv"
portfolio = pd.read_csv(os.path.join(folder, file_portfolio))
shock_data = pd.read_csv(os.path.join(folder, file_shocks))
portfolio.rename(columns={'Regione': 'REGION'}, inplace=True)
shock_data = shock_data[(shock_data['MODEL'] == model) & (shock_data['SCENARIO'] == scenario)]
shock_data = shock_data.round(3)
print (portfolio.head())
print (shock_data.head())



   Titolo     Attivi    Passivi  LGD  REGION
0       1  56.410642  51.768305  0.6  EUROPE
1       2  61.680751  50.518245  0.6  EUROPE
2       3  58.223936  47.087378  0.6  EUROPE
3       4  60.200186  52.684805  0.6  EUROPE
4       5  58.910942  50.868502  0.6  EUROPE
    MODEL   SCENARIO  REGION                              VARIABLE   2025  \
5   WITCH  LIMITS450  EUROPE                   Primary Energy|Coal -0.565   
15  WITCH  LIMITS450  EUROPE                    Primary Energy|Gas -0.449   
25  WITCH  LIMITS450  EUROPE                    Primary Energy|Oil -0.223   
35  WITCH  LIMITS450  EUROPE  Secondary Energy|Electricity|Biomass  0.745   
45  WITCH  LIMITS450  EUROPE     Secondary Energy|Electricity|Coal -0.752   

     2030   2035    2040    2045    2050  
5  -0.614 -0.656  -0.680  -0.699  -0.716  
15 -0.504 -0.547  -0.593  -0.640  -0.687  
25 -0.347 -0.493  -0.620  -0.712  -0.797  
35  4.053  8.427  15.779  28.055  50.269  
45 -0.827 -0.888  -0.920  -0.943  -0.959  


In [32]:
#To compute Credit Risk, granularity add-on by Gordy:

SEED_CREDIT = 42
np.random.seed(SEED_CREDIT)

random_values = gamma.rvs(k, scale=theta, size=num_simulations)

percentile_99_5 = np.percentile(random_values, CONFIDENCE_LEVEL * 100)

rating = ['A', 'BBB', 'BB', 'B', 'CCC']
p = np.array([0.06, 0.20, 1.25, 6.25, 17.50])
w = np.array([1.011, 0.836, 0.602, 0.415, 0.295])

rating_table = pd.DataFrame({
    'Rating': rating,
    'p_mean': p,
    'w': w
})

rating_table['mu_x'] = E_LGD * rating_table['p_mean'] * (1 + rating_table['w'] * (percentile_99_5 - 1))

rating_table['Beta'] = (1 / (2 * lambda_)) * (lambda_ ** 2 + eta ** 2) * (
        (1 / sigma2) * (1 + (sigma2 - 1) / percentile_99_5) *
        (percentile_99_5 + (1 - rating_table['w']) / rating_table['w']) - 1)

n_values = [5000, 2000, 1000, 500, 200, 100]
column_names = ['VaR5000', 'VaR2000', 'VaR1000', 'VaR500', 'VaR200', 'VaR100']

for i, n in enumerate(n_values):
    rating_table[column_names[i]] = rating_table['mu_x'] + (rating_table['Beta'] / n) * 100

rating_table = rating_table.drop(columns=['Beta'])

print(rating_table)


  Rating  p_mean      w       mu_x    VaR5000    VaR2000    VaR1000  \
0      A    0.06  1.011   0.350179   0.366652   0.391361   0.432544   
1    BBB    0.20  0.836   0.982523   0.999404   1.024725   1.066926   
2     BB    1.25  0.602   4.596882   4.614678   4.641371   4.685861   
3      B    6.25  0.415  16.815457  16.834726  16.863629  16.911802   
4    CCC   17.50  0.295  35.998958  36.020156  36.051953  36.104948   

      VaR500     VaR200     VaR100  
0   0.514910   0.762006   1.173834  
1   1.151329   1.404539   1.826554  
2   4.774839   5.041775   5.486667  
3  17.008146  17.297179  17.778900  
4  36.210938  36.528908  37.058858  


In [33]:
#Build the portfolio for simulating climate exposure: 
np.random.seed(SEED_CREDIT) 
num_bonds = len(portfolio)
portfolio['LGD_j'] = np.random.gamma(shape=lambda_, scale=eta, size=num_bonds)
portfolio = portfolio.round(3)
regions = ['NORTH_AM', 'EUROPE', 'ASIA', 'AFRICA', 'LATIN_AM', 'REST_WORLD']
available_regions = portfolio['REGION'].unique().tolist()
years = ['2025', '2030', '2035', '2040', '2045', '2050']
agg_dict = {year: 'max' for year in years}
max_shock_by_region = shock_data.groupby(['REGION', 'MODEL', 'SCENARIO'], as_index=False).agg(agg_dict)
rename_dict = {year: f'Max_Shock_{year}' for year in years}
max_shock_by_region.rename(columns=rename_dict, inplace=True)

all_allowed_sectors = []
for sector_cat, percentage in max_percentages.items():
    if percentage > 0:
        all_allowed_sectors.extend(sectors_data[sector_cat])

all_sims_list = []

print(f"🎲 Simulations starting ({num_simulations})...")

for sim in tqdm(range(1, num_simulations + 1)):
    temp_df = portfolio[['Titolo', 'REGION', 'LGD_j']].copy()
    temp_df['simulation_num'] = sim
    temp_df['sector'] = [random.sample(all_allowed_sectors, min(3, len(all_allowed_sectors))) 
                         for _ in range(len(temp_df))]
    temp_df = temp_df.explode('sector')
    all_sims_list.append(temp_df)
results_df = pd.concat(all_sims_list).reset_index(drop=True)

print(f"✅ results_df creato con {len(results_df)} righe.")

# Weight generation (see transition shock formula based on GVA)
results_df['weight'] = np.random.randint(1, 100, size=len(results_df))
group_sums = results_df.groupby(['simulation_num', 'Titolo'])['weight'].transform('sum')
results_df['weight'] = np.round((results_df['weight'] / group_sums) * 100).astype(int)
current_sums = results_df.groupby(['simulation_num', 'Titolo'])['weight'].transform('sum')
difference = 100 - current_sums
first_row_mask = ~results_df.duplicated(subset=['simulation_num', 'Titolo'])
results_df.loc[first_row_mask, 'weight'] += difference[first_row_mask]
results_df['weight'] = results_df['weight'].clip(lower=0) #This guarantees that we don't have negative weights

print(f"✅ Weights generated and verified for {len(results_df)} rows.")
print(results_df.head())

🎲 Simulations starting (10000)...


100%|██████████| 10000/10000 [00:09<00:00, 1071.77it/s]


✅ results_df creato con 3000000 righe.
✅ Weights generated and verified for 3000000 rows.
   Titolo  REGION  LGD_j  simulation_num                            sector  \
0       1  EUROPE  0.035               1               Primary Energy|Coal   
1       1  EUROPE  0.035               1  Secondary Energy|Electricity|Oil   
2       1  EUROPE  0.035               1  Secondary Energy|Electricity|Gas   
3       2  EUROPE  0.165               1                Primary Energy|Gas   
4       2  EUROPE  0.165               1                Primary Energy|Oil   

   weight  
0      17  
1      34  
2      49  
3      34  
4      27  


In [34]:
# Transition shocks computation
merged_shocks = results_df.merge(
    shock_data, 
    left_on=['REGION', 'sector'], 
    right_on=['REGION', 'VARIABLE'], 
    how='inner'
)
years_str = [str(y) for y in range(2025, 2055, 5)]

for yr in years_str:
    merged_shocks[f'weighted_shock_{yr}'] = (merged_shocks[yr] * merged_shocks['weight']) / 100
weighted_cols = [f'weighted_shock_{yr}' for yr in years_str]
total_shocks = merged_shocks.groupby(
    ['simulation_num', 'Titolo', 'MODEL', 'SCENARIO', 'REGION', 'LGD_j'], 
    as_index=False
)[weighted_cols].sum()
total_shocks.rename(columns={f'weighted_shock_{yr}': f'shock_tot_{yr}' for yr in years_str}, inplace=True)
max_cols = [f'Max_Shock_{yr}' for yr in years_str]
total_shocks = total_shocks.merge(
    max_shock_by_region[['REGION', 'MODEL', 'SCENARIO'] + max_cols],
    on=['REGION', 'MODEL', 'SCENARIO'],
    how='left'
)
total_shocks.rename(columns={f'Max_Shock_{yr}': f'max_shock_{yr}' for yr in years_str}, inplace=True)

print(f"✅ Shock computed across all issuers.")
display(total_shocks.head())

✅ Shock computed across all issuers.


,simulation_num,Titolo,MODEL,SCENARIO,REGION,LGD_j,shock_tot_2025,shock_tot_2030,shock_tot_2035,shock_tot_2040,shock_tot_2045,shock_tot_2050,max_shock_2025,max_shock_2030,max_shock_2035,max_shock_2040,max_shock_2045,max_shock_2050
0,1,1,WITCH,LIMITS450,EUROPE,0.035,-0.40335,-0.47834,-0.52937,-0.58846,-0.64849,-0.70258,1.404,4.053,8.427,15.779,28.055,50.269
1,1,2,WITCH,LIMITS450,EUROPE,0.165,-0.50615,-0.58758,-0.66541,-0.72782,-0.77761,-0.82278,1.404,4.053,8.427,15.779,28.055,50.269
2,1,3,WITCH,LIMITS450,EUROPE,0.006,-0.30562,-0.40623,-0.49961,-0.59252,-0.67141,-0.74300,1.404,4.053,8.427,15.779,28.055,50.269
3,1,4,WITCH,LIMITS450,EUROPE,0.001,-0.46336,-0.49922,-0.52487,-0.56117,-0.60765,-0.65783,1.404,4.053,8.427,15.779,28.055,50.269
4,1,5,WITCH,LIMITS450,EUROPE,0.094,-0.47222,-0.51912,-0.55546,-0.59444,-0.63694,-0.68052,1.404,4.053,8.427,15.779,28.055,50.269


In [35]:
years_list = list(range(2025, 2055, 5))

# Climate transition risk losses calculation (Delta V) - Vectorized
for year in years_list:
    s_tot = f'shock_tot_{year}'
    s_max = f'max_shock_{year}'
    dv_col = f'delta_v_{year}'
    total_shocks[dv_col] = - (total_shocks['LGD_j'] * total_shocks[s_tot]) / (total_shocks[s_max] + 1)
delta_v_cols = [f'delta_v_{year}' for year in years_list]

aggregated_losses = total_shocks.groupby(
    ['simulation_num', 'MODEL', 'SCENARIO'], 
    as_index=False
)[delta_v_cols].sum()

rename_dict = {f'delta_v_{year}': f'sum_delta_v_{year}' for year in years_list}
aggregated_losses.rename(columns=rename_dict, inplace=True)

display(aggregated_losses.head())

,simulation_num,MODEL,SCENARIO,sum_delta_v_2025,sum_delta_v_2030,sum_delta_v_2035,sum_delta_v_2040,sum_delta_v_2045,sum_delta_v_2050
0,1,WITCH,LIMITS450,2.333815,1.323095,0.803170,0.499795,0.312940,0.189431
1,2,WITCH,LIMITS450,2.427784,1.372297,0.833973,0.516862,0.321795,0.193805
2,3,WITCH,LIMITS450,2.194468,1.272933,0.788744,0.496961,0.312697,0.189902
3,4,WITCH,LIMITS450,2.537678,1.376654,0.815967,0.497228,0.307377,0.184851
4,5,WITCH,LIMITS450,2.525512,1.409278,0.849832,0.523621,0.325104,0.195533


In [ ]:
# Compute the transition risk add-on by applying a 99.5 percentile VaR to the empirical distribution of climate losses and discounting back according to Battiston model.
years_list = list(range(2025, 2055, 5))
loss_cols = [f'sum_delta_v_{year}' for year in years_list]
percentile_results = aggregated_losses.groupby(['MODEL', 'SCENARIO'])[loss_cols].quantile(CONFIDENCE_LEVEL).reset_index()
rename_percentile = {col: f'percentile_99.5_{year}' for col, year in zip(loss_cols, years_list)}
percentile_results.rename(columns=rename_percentile, inplace=True)

adjusted_table = percentile_results.copy()

for year in years_list:
    T = year - 2025
    discount = np.exp(-risk_free_rate * T)
    
    perc_col = f'percentile_99.5_{year}'
    adj_col = f'adjusted_{year}'
    adjusted_table[adj_col] = adjusted_table[perc_col] * discount
    adjusted_table[adj_col] = adjusted_table[adj_col].apply(lambda x: f"{x:.3f}%")

cols_to_keep = ['MODEL', 'SCENARIO'] + [f'adjusted_{year}' for year in years_list]
adjusted_table = adjusted_table[cols_to_keep]

print(adjusted_table)

   MODEL   SCENARIO adjusted_2025 adjusted_2030 adjusted_2035 adjusted_2040  \
0  WITCH  LIMITS450        2.611%        1.236%        0.636%        0.336%   

  adjusted_2045 adjusted_2050  
0        0.179%        0.093%  
